In [ ]:
import numpy as np
import pandas as pd
from scipy import stats



---

## Task 1: Formulating Hypotheses

**Scenario:** You believe that people spend more time on Instagram than on YouTube daily.

* **Null Hypothesis ($H_0$):** There is no significant difference in the average daily time spent on Instagram compared to YouTube, or people spend less time on Instagram than on YouTube daily.

$$\mu_{\text{Instagram}} \le \mu_{\text{YouTube}} \quad \text{or} \quad \mu_{\text{Instagram}} - \mu_{\text{YouTube}} \le 0$$

* **Alternative Hypothesis ($H_1$):** People spend significantly more time on Instagram than on YouTube daily (One-tailed test).

$$\mu_{\text{Instagram}} > \mu_{\text{YouTube}} \quad \text{or} \quad \mu_{\text{Instagram}} - \mu_{\text{YouTube}} > 0$$

```

```

## Task 2: Two-Sample t-Test (Zomato AOV Experiment)

**Given Data:**
- **Group A (New Home Page):** Sample size (n_A) = 30, Mean (A) = ₹350, Standard Deviation (A) = ₹50
- **Group B (Old Home Page):** Sample size (n_B) = 30, Mean (B) = ₹320, Standard Deviation (B) = ₹50
- **Significance Level (alpha):** 0.05

**Hypotheses:**
- (H_0: mu_A = mu_B) (The new home page does not change the average order value)
- (H_1: mu_A != mu_B) (The new home page significantly changes the average order value)

In [ ]:

# Summary statistics table
summary_data = {
    "Group": ["Group A (New Home Page)", "Group B (Old Home Page)"],
    "Sample Size (n)": [30, 30],
    "Mean AOV (₹)": [350.0, 320.0],
    "Std Dev (₹)": [50.0, 50.0]
}

df_summary = pd.DataFrame(summary_data)
print("--- Experiment Summary ---")
print(df_summary.to_string(index=False))

# Variables
n_A, mean_A, std_A = 30, 350.0, 50.0
n_B, mean_B, std_B = 30, 320.0, 50.0
alpha = 0.05

# 1. Calculation using NumPy formulas
pooled_std = np.sqrt(((n_A - 1) * (std_A ** 2) + (n_B - 1) * (std_B ** 2)) / (n_A + n_B - 2))
standard_error = pooled_std * np.sqrt((1 / n_A) + (1 / n_B))
t_stat_numpy = (mean_A - mean_B) / standard_error
degrees_of_freedom = n_A + n_B - 2

# 2. Scipy two-sample t-test from summary stats
t_stat, p_value = stats.ttest_ind_from_stats(
    mean1=mean_A, std1=std_A, nobs1=n_A,
    mean2=mean_B, std2=std_B, nobs2=n_B,
    equal_var=True
)

print("\n--- Test Results ---")
print(f"Degrees of Freedom: {degrees_of_freedom}")
print(f"t-statistic:        {t_stat:.4f}")
print(f"p-value:            {p_value:.4f}")
print(f"Significance Level: {alpha}")

# Decision rule
if p_value < alpha:
    print("\nDecision: Reject the Null Hypothesis (H0)")
    print("Conclusion: There is a statistically significant difference in Average Order Value between the two home pages at the 0.05 level.")
else:
    print("\nDecision: Fail to Reject the Null Hypothesis (H0)")
    print("Conclusion: There is no statistically significant difference in Average Order Value at the 0.05 level.")

## Task 3: A/B Testing Setup for an App Feature

### Feature: Redesigned Payment Button in Paytm

* **App & Feature:** Paytm — Redesigning the final "Pay Now" CTA button to be full-width, sticky at the bottom of the screen, with dynamic cash-back reminder text.
* **Control Group (Group A):** 50% of randomly selected users during checkout who see the standard, static payment button.
* **Variant Group (Group B):** 50% of randomly selected users during checkout who see the redesigned sticky CTA button.
* **Primary Metric to Measure:**

$$\text{Payment Conversion Rate (PCR)} = \frac{\text{Number of Successful Payments}}{\text{Total Users Reaching Checkout Screen}} \times 100$$

Hypotheses:Null Hypothesis ($H_0$): The redesigned payment button does not increase the payment conversion rate compared to the existing button ($p_{\text{variant}} \le p_{\text{control}}$).

Alternative Hypothesis ($H_1$): The redesigned payment button leads to a significantly higher payment conversion rate than the existing button ($p_{\text{variant}} > p_{\text{control}}$).



## Task 4: Flipkart Wishlist Button A/B Test Analysis

### 1. Is this difference likely to be significant or just due to chance?
* **Observation:** Out of 1,000 total interactions, 520 clicked the new button (52%) versus 480 who clicked the old button (48%).
* **Analysis:** A split of 52% vs. 48% across a sample of 1,000 represents only a 4% absolute difference (40 clicks). In random sampling from an evenly matched population, standard binomial variation naturally produces fluctuations of 2%to  3%. Therefore, without running complex calculations, this modest difference is **very likely due to random chance** rather than a genuine shift in user behavior.

### 2. How does the p-value help decide?
* **The Role of the p-value:** The p-value quantifies the probability of observing an outcome at least as extreme as 520 vs. 480 clicks, assuming the null hypothesis (\(H_0\): both buttons perform identically) is completely true.
* **Decision Framework:**
  * If the calculated **\(p-value < 0.05\)**, the probability of seeing this gap purely by chance is less than 5%. In that case, we would reject \(H_0\) and declare the new design genuinely better.
  * If the calculated **\(p-value > 0.05\)**, the observed gap is consistent with routine random variation. We fail to reject \(H_0\) and conclude there is insufficient evidence to adopt the new button.

In [ ]:
# Verification of Task 4 using pandas and statsmodels/scipy proportions test


# Data representation
flipkart_data = pd.DataFrame({
    "Variant": ["New Button", "Old Button"],
    "Clicks": [520, 480],
    "Total Users": [500, 500],  # Equal split allocation
    "Click Rate (%)": [52.0, 48.0]
})

print("--- Flipkart A/B Test Data ---")
print(flipkart_data.to_string(index=False))

# Two-proportion z-test
count = np.array([520, 480])
nobs = np.array([500, 500])  # If evaluated as separate 500-user cohorts

# Chi-squared goodness of fit test against an expected 50/50 split (500 vs 500)
observed = np.array([520, 480])
expected = np.array([500, 500])

chi2_stat, p_val = stats.chisquare(f_obs=observed, f_exp=expected)

print("\n--- Chi-Square Goodness-of-Fit Test ---")
print(f"Chi2 Statistic: {chi2_stat:.4f}")
print(f"p-value:        {p_val:.4f}")

if p_val >= 0.05:
    print(f"\nResult: p-value ({p_val:.4f}) >= 0.05 -> Difference is NOT statistically significant; likely due to chance.")
else:
    print(f"\nResult: p-value ({p_val:.4f}) < 0.05 -> Statistically significant.")